In [0]:
%pip install loguru

In [0]:
from loguru import logger
import sys

logger.remove()  # Remove the default logger

logger.add(
    sys.stderr,
    level="INFO",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}"
)


In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("merge_keys", "")

table_name = dbutils.widgets.get("table_name").strip()
logger.debug(f"table_name: {table_name}")
landing_timestamp = dbutils.widgets.get("landing_timestamp").strip()
load_type = dbutils.widgets.get("load_type").strip().upper()
merge_keys_input = dbutils.widgets.get("merge_keys").strip()

In [0]:
if not table_name:
    raise ValueError("table_name can not be empty")

if load_type not in ("APPEND", "FULL", "MERGE"):
    raise ValueError("load_type must be either APPEND, FULL or OVERWRITE")

In [0]:
bronze_table = f"`pulselink-catalog`.bronze.{table_name}"
logger.debug(f"bronze_table: {bronze_table}")
silver_table = f"`pulselink-catalog`.silver.{table_name}"
logger.debug(f"silver_table: {silver_table}")

In [0]:
bronze_df = spark.read.table(bronze_table)

In [0]:
logger.info(f"Starting Silver load | table: {table_name} | load_type: {load_type}")

if load_type == "FULL":
    
    logger.info("FULL load: Adding insert_timestamp and update_timestamp columns")
    updated_df = (
        bronze_df
        .withColumn("insert_timestamp", current_timestamp())
        .withColumn("update_timestamp", current_timestamp())
    )
    
    logger.info(f"FULL load: Overwriting Silver table {silver_table}")
    updated_df.write.mode("overwrite").saveAsTable(silver_table)
    logger.info(f"FULL load: Successfully written to {silver_table}")
    

elif load_type == "MERGE":

    logger.info(f"MERGE load: landing_timestamp={landing_timestamp}, merge_keys={merge_keys_input}")

    if not landing_timestamp:
            raise ValueError(
                "landing_timestamp is required for MERGE"
            )

    if not merge_keys_input:
            raise ValueError(
                "merge_keys is required for MERGE"
            )

    source_df = bronze_df.where(col("landing_timestamp") == lit(landing_timestamp))
    logger.info(f"MERGE load: Filtered Bronze for landing_timestamp={landing_timestamp}")

    # Create Silver on the first load.
    if not spark.catalog.tableExists(f"`pulselink-catalog`.silver.{table_name}"):
        logger.info(f"Silver table {silver_table} does not exist — creating it (first load)")
        (
            source_df
            .write
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        logger.info(f"Silver table created for {table_name}")

    else:

        logger.info(f"Silver table {silver_table} already exists — proceeding with MERGE")

        merge_keys = [ key.strip() for key in merge_keys_input.split(",") if key.strip()]
        logger.info(f"Parsed merge keys: {merge_keys}")

        if not merge_keys:
                raise ValueError("At least one merge key is required")

        missing_keys = [
                key for key in merge_keys
                if key not in bronze_df.columns
            ]

        if missing_keys:
                raise ValueError(
                    f"Merge keys not found in Bronze: {missing_keys}"
                )

        if "insert_timestamp" not in source_df.columns:
            source_df = source_df.withColumn("insert_timestamp", current_timestamp())
            logger.info("Added insert_timestamp column to source")

        source_df = source_df.withColumn(
            "update_timestamp", current_timestamp()
        )
        logger.info("Added update_timestamp column to source")

        source_df.createOrReplaceTempView("incremental_source")
        logger.info("Created temp view: incremental_source")

        merge_condition = " AND ".join(
            [
                f"target.`{key}` = source.`{key}`"
                for key in merge_keys
            ]
        )

        logger.info(f"Merge condition: {merge_condition}")

        update_columns = [
            column_name
            for column_name in source_df.columns
            if column_name != "insert_timestamp"
        ]

        logger.info(f"Update columns: {update_columns}")

        update_set = ", ".join(
            [
                f"target.`{column_name}` = "
                f"source.`{column_name}`"
                for column_name in update_columns
            ]
        )

        logger.info(f"Update set: {update_set}")

        insert_columns = ", ".join(
            [f"`{column_name}`" for column_name in source_df.columns]
        )

        logger.info(f"Insert columns: {insert_columns}")


        insert_values = ", ".join(
            [f"source.`{column_name}`"
             for column_name in source_df.columns]
        )

        merge_sql = f"""
            MERGE INTO {silver_table} AS target
            USING incremental_source AS source
            ON {merge_condition}

            WHEN MATCHED THEN
                UPDATE SET {update_set}

            WHEN NOT MATCHED THEN
                INSERT ({insert_columns})
                VALUES ({insert_values})
        """

        logger.debug(f"Merge SQL: {merge_sql}")
        record_count = source_df.count()
        logger.info(f"Total records to merge: {record_count}")


In [0]:
if load_type == "FULL":
    record_count = spark.table(
        silver_table
    ).count()
else:
    record_count = source_df.count()

dbutils.notebook.exit(str(record_count))